In [ ]:
!pip install unsloth trl datasets transformers accelerate kagglehub pandas langchain-groq

In [ ]:
import json
import pandas as pd
from pathlib import Path
import kagglehub
from kagglehub import KaggleDatasetAdapter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from concurrent.futures import ThreadPoolExecutor
import os

# --- Setup ---
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE" # <--- PASTE KEY HERE

DATASET_SIZE = 500 # Groq is fast enough to handle 500+ easily
MAX_WORKERS = 8 # Tune this based on API limits and machine resources
OUTPUT_FILE = "data/resume_dataset.json"

SYSTEM_PROMPT = """You are a precise document parsing agent.
Your ONLY job is to read the provided document text and return a single valid JSON object.
Rules:
- Return ONLY the JSON object. No explanation, markdown, or text outside the JSON.
- If a field is not present in the document, set it to null.
- For list fields (skills, experience, education), use an empty list [] if nothing is found.

Output schema:
{
  \"name\": string or null,
  \"email\": string or null,
  \"phone\": string or null,
  \"website\": string or null,
  \"skills\": [string, ...],
  \"experience\": [{\"title\": string, \"company\": string, \"duration\": string}, ...],
  \"education\": [{\"degree\": string, \"institution\": string, \"year\": string}, ...]
}"""

print("Downloading Kaggle Dataset...")
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "snehaanbhawal/resume-dataset", "Resume.csv")
raw_resumes = df.sample(n=DATASET_SIZE, random_state=42)['Resume_str'].tolist()

# Initialize blazing-fast Groq Llama 3
teacher_llm = ChatGroq(model="llama3-8b-8192", temperature=0.0, model_kwargs={"response_format": {"type": "json_object"}})
prompt = ChatPromptTemplate.from_messages([('system', SYSTEM_PROMPT), ('human', '{text}')])
chain = prompt | teacher_llm

def process_resume(text):
    clean_text = str(text).replace("\r", "").strip()[:4000]
    try:
        result = chain.invoke({"text": clean_text}).content
        json.loads(result) # Validate it is valid JSON
        return {"instruction": SYSTEM_PROMPT, "input": clean_text, "output": result}
    except Exception as e:
        print(f"Skipping one resume due to parse error: {e}")
        return None

print(f"Generating ground truth for {DATASET_SIZE} resumes concurrently...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor: # Process concurrently
    results = list(executor.map(process_resume, raw_resumes))

dataset_records = [r for r in results if r is not None]

Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(dataset_records, f, indent=2)

print(f"Success! Annotated {len(dataset_records)} high-quality examples.")


In [ ]:
from datasets import Dataset

with open("data/resume_dataset.json", encoding="utf-8") as f:
    raw = json.load(f)

def format_example(ex):
    return {
        "text": (
            f"### System:\n{ex['instruction']}\n\n"
            f"### Document:\n{ex['input']}\n\n"
            f"### Extracted JSON:\n{ex['output']}"
        )
    }

hf_dataset = Dataset.from_list(raw).map(format_example, remove_columns=list(raw[0].keys()))
print("Dataset formatted for SmolLM2 training.")


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="HuggingFaceTB/SmolLM2-360M",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

cuda_available = torch.cuda.is_available()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True, # <--- Highly optimized: Combines short resumes, can significantly speed up training
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=cuda_available and not torch.cuda.is_bf16_supported(),
        bf16=cuda_available and torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="./outputs",
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer.train()


In [ ]:
from huggingface_hub import notebook_login
# Paste your Hugging Face WRITE token when prompted
notebook_login()


In [ ]:
HF_REPO = "your_username/SmolLM2-360M-AgentHire-Extractor" # <--- UPDATE WITH YOUR HF USERNAME

print("Pushing standard weights to Hub...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print("Pushing GGUF to Hub (for Ollama)...")
model.push_to_hub_gguf(
    HF_REPO,
    tokenizer,
    quantization_method="q4_k_m",
)

print("Done")
